In [1]:
import requests
from bs4 import BeautifulSoup
import time
import re
from pymongo import MongoClient
from collections import OrderedDict

# --- CONFIGURACIÓN ---
NOMBRE_GRUPO = "G9-AgroTech"
NOMBRE_INTEGRANTE = "Alejandro Nuñez"
NOMBRE_ETIQUETA = "Naranjas"
URL_OBJETIVO = "https://www.indexmundi.com/es/precios-de-mercado/?mercancia=naranjas&meses=240&moneda=clp"

meses_completos = {
    "ene.": "enero", "feb.": "febrero", "mar.": "marzo", "abr.": "abril",
    "may.": "mayo", "jun.": "junio", "jul.": "julio", "ago.": "agosto",
    "sep.": "septiembre", "oct.": "octubre", "nov.": "noviembre", "dic.": "diciembre"
}

# --- PASO 1: SCRAPING Y FILTRADO (>= 2020) ---
datos_finales = []
try:
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(URL_OBJETIVO, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    filas = soup.find_all('tr')
    
    print("--- 1. Iniciando extracción (Filtrando desde 2020 en adelante) ---")
    
    for fila in filas:
        celdas = fila.find_all(['td', 'th'])
        if len(celdas) >= 2:
            texto_fila = [c.get_text(strip=True) for c in celdas]
            
            match_fecha = re.search(r'([a-z]{3}\.?)\s+(\d{4})', texto_fila[0].lower())
            if match_fecha:
                abr_mes = match_fecha.group(1)
                if not abr_mes.endswith('.'): abr_mes += '.'
                anio_str = match_fecha.group(2)
                anio_int = int(anio_str)
                
                if anio_int >= 2020:
                    precio_raw = texto_fila[1]
                    variacion_raw = texto_fila[2] if len(texto_fila) > 2 else "-"
                    
                    if precio_raw and any(char.isdigit() for char in precio_raw):
                        nombre_mes = meses_completos.get(abr_mes, abr_mes)
                        
                        item = OrderedDict()
                        item["grupo"] = NOMBRE_GRUPO
                        item["Integrante"] = NOMBRE_INTEGRANTE
                        item["Etiqueta"] = NOMBRE_ETIQUETA
                        item["Año"] = anio_str
                        item["Mes"] = nombre_mes
                        item["valor_sucio"] = precio_raw
                        item["variacion_sucia"] = variacion_raw
                        datos_finales.append(item)

    print(f"✅ Extracción finalizada: {len(datos_finales)} registros encontrados desde 2020.")
except Exception as e:
    print(f"❌ Error en Scraping: {e}")

# --- PASO 2: CONEXIÓN Y CARGA ---
if datos_finales:
    host_docker = 'mongodb://bigdata_mongodb:27017/'
    try:
        client = MongoClient(host_docker, serverSelectionTimeoutMS=5000)
        client.server_info()
        db = client['previa_proyecto']
        coleccion = db['naranjas']
        
        exitos = 0
        for dato in datos_finales:
            try:
                v_limpio = dato['valor_sucio'].replace('.', '').replace(',', '.')
                dato['Precio'] = float(v_limpio)
                
                v_var = dato['variacion_sucia'].replace('%', '').replace(',', '.').strip()
                if v_var == "-" or not v_var or v_var == "0.00":
                    dato['Variacion decimal'] = 0.0
                else:
                    dato['Variacion decimal'] = float(v_var)
                
                del dato['valor_sucio'], dato['variacion_sucia']
                dato["Fecha de captura"] = time.strftime("%Y-%m-%d %H:%M:%S")
                
                coleccion.insert_one(dato)
                exitos += 1
            except:
                continue
            
        print("\n" + "="*30)
        print(f"CARGA EXITOSA (2020-Presente)")
        print(f"Registros guardados: {exitos}")
        print("="*30)
        client.close()
    except Exception as e:
        print(f"❌ Error Mongo: {e}")
else:
    print("⚠️ No se encontraron registros que cumplan el criterio (>= 2020).")


--- 1. Iniciando extracción (Filtrando desde 2020 en adelante) ---
✅ Extracción finalizada: 66 registros encontrados desde 2020.

CARGA EXITOSA (2020-Presente)
Registros guardados: 66
